In [ ]:
# ==============================================================================
# 1. Importacion de Librerias y Carga del Modelo
# ==============================================================================
import pandas as pd
import numpy as np
import joblib
import os

print("Preparando entorno de Virtual Screening...")

# Cargar el modelo guardado desde el Cuaderno 02
ruta_modelo = "../models/best_rf_pampa.pkl"

try:
    best_rf = joblib.load(ruta_modelo)
    print(f" Modelo Random Forest cargado exitosamente.")
    
    # Extraer los nombres exactos de las 11 variables que el modelo exige
    features_modelo = best_rf.feature_names_in_
    print(f" El modelo requiere exactamente estas {len(features_modelo)} variables:")
    print(list(features_modelo))
    
except FileNotFoundError:
    print(f" ERROR: No se encontro el modelo en {ruta_modelo}.")

In [ ]:
# ==============================================================================
# 2. Motor Central de Cribado Virtual
# ==============================================================================

def ejecutar_cribado(file_desc, file_fing, archivo_salida):
    """
    Toma los descriptores, fingerprints, los cruza, filtra las 11 variables,
    predice la probabilidad con el modelo y guarda un CSV limpio.
    """
    print(f"Iniciando procesamiento para: {os.path.basename(archivo_salida)}")
    
    try:
        # 1. Carga y limpieza
        df_d = pd.read_csv(file_desc, sep='\t')
        df_f = pd.read_csv(file_fing, sep='\t')
        
        if 'NAME' in df_d.columns: df_d['NAME'] = df_d['NAME'].astype(str).str.strip()
        if 'NAME' in df_f.columns: df_f['NAME'] = df_f['NAME'].astype(str).str.strip()
        
        # 2. Fusion
        df_total = pd.merge(df_d, df_f, on='NAME', how='inner')
        print(f"   -> FusiÃƒÂ³n correcta: {len(df_total)} molÃƒÂ©culas encontradas.")
        
        # 3. Filtrado de las 11 variables
        X_input = pd.DataFrame()
        for col in features_modelo:
            if col in df_total.columns:
                X_input[col] = df_total[col]
            else:
                X_input[col] = 0 # Relleno de seguridad
                
        # Limpiar infinitos y nulos
        X_input = X_input.fillna(0).replace([np.inf, -np.inf], 0)
        
        # 4. Prediccion
        probs = best_rf.predict_proba(X_input)[:, 1]
        
        # 5. Construccion del DataFrame Final
        df_final = df_total[['NAME']].copy()
        df_final = df_final.rename(columns={'NAME': 'ID_Molecula'})
        df_final['Probabilidad_Permeable'] = probs
        
        # Agregamos las 11 variables para el reporte
        for col in features_modelo:
            df_final[col] = X_input[col]
            
        # Ordenar de mayor a menor probabilidad
        df_final = df_final.sort_values(by='Probabilidad_Permeable', ascending=False)
        
        # 6. Guardado
        os.makedirs(os.path.dirname(archivo_salida), exist_ok=True)
        df_final.to_csv(archivo_salida, index=False)
        print(f"Archivo guardado: {archivo_salida}")
        
        return df_final

    except Exception as e:
        print(f" Error en el proceso: {e}")
        return None

In [ ]:
# ==============================================================================
# 3. Ejecucion de las Fases de Cribado
# ==============================================================================

# FASE 1: PRE-CRIBADO (Las 5 molÃƒÂ©culas de prueba)
# Nota: AsegÃƒÂºrate de que estas rutas sean correctas en tu PC
ruta_desc_5 = r"../archive/source_material/qsar_inputs/calculos-screeningg.txt"
ruta_fing_5 = r"../archive/source_material/qsar_inputs/calculos-screening-fingerr.txt"
salida_5 = "../results/screening/Pre_Cribado_5_Moleculas.csv"

df_precribado = ejecutar_cribado(ruta_desc_5, ruta_fing_5, salida_5)

if df_precribado is not None:
    print("\nVista previa del Pre-Cribado (Top 5):")
    display(df_precribado[['ID_Molecula', 'Probabilidad_Permeable', 'LOGPcons', 'MACCSFP125']].head())

print("-" * 60)

# FASE 2: DRUGBANK COMPLETO
# El cribado DrugBank completo de la tesis se conserva como resultado final en:
# ../results/screening/DrugBank_Candidatos_Tesis.csv
#
# Los archivos fuente usados para reconstruirlo se mantienen en ../archive/source_material/qsar_inputs/:
# - Drugbank.txt / drugbank.xlsx
# - fingerprints.txt
# - allCOR3D.sdf
# - calculos-screening*.txt
#
# No se regenera aqui para evitar mezclar el cribado completo (>10000 compuestos)
# con corridas exploratorias antiguas de 34 moleculas.


In [ ]:
# ==============================================================================
# 4. Evaluacion de Drogabilidad (Lipinski) y Visualizacion Estructural 2D
# ==============================================================================
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors, Draw, AllChem
from IPython.display import display
import os

print("Inicializando modulo de Quimioinformatica y cruce de datos...")

# 1. Configurar rutas
# Ajusta las rutas segun donde tengas descargados los archivos
archivo_resultados = "../results/screening/Pre_Cribado_5_Moleculas.csv" 
archivo_SDF = r"../archive/source_material/qsar_inputs/allCOR3D.sdf"
archivo_traductor = r"../archive/source_material/qsar_inputs/Lista_DrugBank_IDs.csv"

df_hits = pd.read_csv(archivo_resultados)

# 2. Crear el "Traductor" de nombres (MoleculeX -> DBXXXXX)
dicc_traductor = {}
try:
    df_nombres = pd.read_csv(archivo_traductor)
    for _, row in df_nombres.iterrows():
        nombre_falso = f"Molecule{int(row['Orden'])}"
        nombre_real = str(row['DrugBank_ID']).strip()
        dicc_traductor[nombre_falso] = nombre_real
    print(f"STATUS: Traductor cargado. {len(dicc_traductor)} equivalencias registradas.")
except FileNotFoundError:
    print(f"ERROR: No se encontro el archivo traductor en: {archivo_traductor}")

# 3. Extraer SMILES del archivo SDF
dicc_smiles = {}
try:
    with open(archivo_SDF, 'r', encoding='utf-8', errors='ignore') as f:
        bloques = f.read().split("$$$$")
        for bloque in bloques:
            lines = bloque.split('\n')
            curr_id, curr_smi = None, None
            
            for j, line in enumerate(lines):
                if "> <DRUGBANK_ID>" in line: 
                    curr_id = lines[j+1].strip()
                if "> <DRUGBANK_CANONICAL_SMILES>" in line:
                    curr_smi = lines[j+1].strip()
            
            if curr_id and curr_smi:
                dicc_smiles[curr_id] = curr_smi
    print(f"STATUS: Analisis SDF completado. {len(dicc_smiles)} estructuras indexadas.")
except FileNotFoundError:
    print(f"ERROR: No se encontro el archivo SDF en: {archivo_SDF}")

# 4. Analisis Fisicoquimico de Candidatos
resultados_quimicos = []
mols_dibujo = []
leyendas = []

for idx, row in df_hits.iterrows():
    id_falso = str(row['ID_Molecula'])
    prob = row['Probabilidad_Permeable']
    
    # 4.1 Traducir ID
    id_real = dicc_traductor.get(id_falso, id_falso) 
    
    # 4.2 Buscar SMILES con el ID real
    smi = dicc_smiles.get(id_real, None)
    
    if smi:
        mol = Chem.MolFromSmiles(smi)
        if mol:
            # Calcular parametros
            mw = Descriptors.MolWt(mol)
            logp = Descriptors.MolLogP(mol)
            hbd = Descriptors.NumHDonors(mol)
            hba = Descriptors.NumHAcceptors(mol)
            
            violaciones = sum([mw > 500, logp > 5, hbd > 5, hba > 10])
            es_drogable = "SI" if violaciones <= 1 else "NO"
            
            resultados_quimicos.append({
                'ID_Original': id_falso,
                'DrugBank_ID': id_real,
                'Probabilidad': prob,
                'MW': round(mw, 2),
                'LogP_RDKit': round(logp, 2),
                'HBD': hbd,
                'HBA': hba,
                'Violaciones_Lipinski': violaciones,
                'Aprobado_Lipinski': es_drogable
            })
            
            # Preparar visualizacion 2D
            AllChem.Compute2DCoords(mol)
            leyenda = f"{id_real}\nProb: {prob:.2f} | Lip: {es_drogable}"
            mol.SetProp("_Name", leyenda)
            mols_dibujo.append(mol)
            leyendas.append(leyenda)
    else:
        print(f"ADVERTENCIA: La estructura para '{id_real}' (originalmente {id_falso}) no se encontro en el SDF.")

# 5. Reporte Final y Graficacion
if resultados_quimicos:
    df_lipinski = pd.DataFrame(resultados_quimicos)
    print("\n--- Resultados del Filtro de Lipinski ---")
    display(df_lipinski[['DrugBank_ID', 'Probabilidad', 'MW', 'LogP_RDKit', 'HBD', 'HBA', 'Aprobado_Lipinski']])
    
    # Exportar
    ruta_reporte = "../results/screening/Reporte_Lipinski_Candidatos.csv"
    os.makedirs(os.path.dirname(ruta_reporte), exist_ok=True)
    df_lipinski.to_csv(ruta_reporte, index=False)
    
    if len(mols_dibujo) > 0:
        print("\n--- Renderizado Estructural 2D ---")
        img_grid = Draw.MolsToGridImage(
            mols_dibujo,
            molsPerRow=4,
            subImgSize=(300, 300),
            legends=leyendas,
            returnPNG=False 
        )
        display(img_grid)
        
        ruta_imagen = "../results/figures/Galeria_Top_Candidatos.png"
        img_grid.save(ruta_imagen)
        print(f"STATUS: Galeria exportada a {ruta_imagen}")